In [ ]:

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [ ]:
import sys
import os

# Go up one level to the project root (add more '..' if you are deeper)
project_root = os.path.abspath('..')
print(f"This is the project_root{project_root}")
if project_root not in sys.path:
    sys.path.append(project_root)
!git clone https://github.com/tom-esplin/midi-generator-model

In [ ]:
!ls
!cd midi-generator-model && ls

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on {}".format(device))
print(torch.version.cuda)
print(torch.__version__)   

In [ ]:
genre = "folk"
from training.prep_dataset import MidiDataset, ContinuousMidiDataset
from pathlib import Path
from miditok import PerTok,TokSequence
#add function to autoselect genre path
#start with small dataset
CHUNK_SIZE = 1000
exp_path = Path("tokenization","saved_tokens","folk-0-04-04-2026_12-22-01")
tokenizer = PerTok(params=Path(exp_path,"tokenizer.json"))
print(f"Loaded Tokenizer Vocab Size: {len(tokenizer)}")

In [ ]:
@torch.no_grad()
def generate(x, model, num_layers, hidden_dim, pred_length, temperature=0.8):
    # 1. Initialize hidden state
    hidden = torch.zeros(num_layers, x.size(0), hidden_dim).to(device)
    # 2. "Warm up" the model with the starting sequence (x)
    # This gets the hidden state synced up with the prompt
    logits, hidden = model(x, hidden)
    
    # Take the very last predicted token from the prompt to start generating
    # shape: [batch, 1]
    current_token = torch.multinomial(torch.softmax(logits[:, -1, :] / temperature, dim=-1), 1)
    
    generated_tokens = [x, current_token]

    # 3. Generation loop
    for _ in range(pred_length):
        # Pass ONLY the single last token and the updated hidden state
        logits, hidden = model(current_token, hidden)
        
        # Apply temperature and softmax to the logits
        # logits shape: [batch, 1, vocab_size]
        probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
        
        # Sample the next token
        next_token = torch.multinomial(probs, 1) # shape: [batch, 1]
        
        generated_tokens.append(next_token)
        current_token = next_token # Use this for the next iteration
        
    # Concatenate all tokens along the sequence dimension (dim=1)
    return torch.cat(generated_tokens, dim=1)

In [ ]:
eval_song = "001013_0.mid"
EVAL_START_TOKEN_IDS = torch.tensor(tokenizer(Path("prepared_data","folk","test",eval_song))[0].ids).unsqueeze(0)
EVAL_START_TOKEN_IDS = EVAL_START_TOKEN_IDS[:,:CHUNK_SIZE].to(device)
def evaluate(model,num_layers, hidden_dim, pred_length, iteration_count):
    """Runs generate function and formats and prints predictions.
    """
    model.eval()
    inference_batch = generate(EVAL_START_TOKEN_IDS, model, num_layers, hidden_dim, pred_length)
    for i,inference_tokens in enumerate(inference_batch):
        list_of_ids = inference_tokens.squeeze().tolist()
        tok_sequence = TokSequence(ids=list_of_ids,are_ids_encoded=True)
        pred_midi = tokenizer.decode([tok_sequence])
        pred_midi.dump_midi(Path(f"test_{iteration_count}.mid"))
    model.train()

In [ ]:
import time
from datetime import datetime
from tqdm import tqdm
import torch
import torch.nn as nn

def train(model, num_layers, hidden_dim, optimizer, dataloader, n_optimization_steps, eval_interval):
    # Label smoothing adds a bit of uncertainty to the model, improving text generation.
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    losses = []
    model.train()
    start_time = time.time()
    data_iter = iter(dataloader)
    
    for step in tqdm(range(n_optimization_steps)):
        try:
            batch = next(data_iter)
        except StopIteration as e:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        batch = batch.to(device)
        hidden = torch.zeros(num_layers, batch.size(0), hidden_dim).to(device)
        
        # Inner loop: iterate over sequence
        for i in tqdm(range(CHUNK_SIZE - 1), leave=False):
            x = batch[:, i:i+1]
            y_truth = batch[:,i+1:i+2]
            
            optimizer.zero_grad()
            y_hat, hidden = model(x, hidden)
            loss = loss_fn(y_hat, y_truth.flatten())
            
            if i == 0:
                print(f"Step {step} Loss: {loss.item()}")
                
            loss.backward()
            optimizer.step()
            hidden = hidden.detach()
            losses.append(loss.item())
            
        # --- FIXED INDENTATION: Run eval and save AFTER the sequence loop finishes ---
        if (step + 1) % eval_interval == 0:
            model.eval()
            print(f"\nStep:{step} Loss {losses[-1]}\n\n")
            evaluate(model, num_layers, hidden_dim, CHUNK_SIZE, step)
            model.train()
            
        if time.time() - start_time >= 360000:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            torch.save(model.state_dict(),'model_{}_{}'.format(timestamp, step))
            torch.save(optimizer.state_dict(),'optimizer_{}_{}'.format(timestamp, step))

    return losses

In [ ]:



from models.gru import GRUModel
from torch.optim import Adam

dataset = MidiDataset(folder_path=Path(exp_path,"test"),chunk_size=CHUNK_SIZE)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
num_layers = 2
hidden_dim = 256
small_model = GRUModel(tokenizer.vocab_size, embedding_dim=256, hidden_dim=hidden_dim, num_layers=num_layers).to(device)

optimizer = Adam(small_model.parameters(), lr=5e-4)


In [ ]:
losses = train(small_model, num_layers, hidden_dim, optimizer, dataloader, n_optimization_steps=2000, eval_interval=100)

In [ ]:
#save the model and optimizer state dicts
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
torch.save(small_model.state_dict(),'models/mode_weights/gru_model_{}_{}'.format(timestamp, 2000))
torch.save(optimizer.state_dict(),'models/mode_weights/gru_optimizer_{}_{}'.format(timestamp, 2000))

In [ ]:
plt.plot(losses)
plt.title("GRU Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.savefig(f"gru_training_loss_{timestamp}.png")
plt.show()

In [ ]:
# generate some music
batch = generate(EVAL_START_TOKEN_IDS, small_model,num_layers,hidden_dim, pred_length=CHUNK_SIZE)
for i,inference_tokens in enumerate(batch):
        list_of_ids = inference_tokens.squeeze().tolist()
        tok_sequence = TokSequence(ids=list_of_ids,are_ids_encoded=True)
        pred_midi = tokenizer.decode([tok_sequence])
        pred_midi.dump_midi(Path(f"test_generated.mid"))